# 15 — SASRec vs Mult-VAE (autoencoder): accuracy + anti-popularity

Runs the **standalone SASRec port** (`book_recsys/models/sequential`) — **no RecBole** — against
the **Mult-VAE autoencoder** and the max-sim / SVD baselines, on the *same users, same protocol*.

**(A) Accuracy** — popularity-matched sampled-negative leave-1-out (the study headline protocol),
on the same 30k / seed-42 / MAX_HIST=100 subsample the SASRec checkpoint trained on. Reproduces
the existing `study_sasrec_vs_maxsim.csv` SASRec column and adds Mult-VAE (α=0).

**(B) Anti-popularity diagnostic** — full-catalog top-10 → mean popularity percentile, catalog
coverage, Gini of recommendation frequency. Lower percentile / higher coverage = less
popularity-biased (more long-tail).

**No RecBole. GPU optional** (auto-uses CUDA on Kaggle; CPU works, just slower). **Attach a dataset
containing:** `SASRec_state.pt`, `SASRec_item_map.json`, `sample.parquet`, `catalog.parquet`,
`embeddings.npy`, `models.joblib`, `multvae_last.pt`.

> If `SASRec_state.pt` isn't in your dataset, generate it once from `SASRec.pth` with
> `python scripts/export_sasrec_state.py` (offline; needs only torch — it strips the RecBole objects out, no RecBole install required).

In [ ]:
# --- Make the FULL book_recsys package importable (Kaggle/Colab) ---
import sys, os, glob, subprocess
try:
    import book_recsys.data  # noqa: F401
except ModuleNotFoundError:
    hits = glob.glob('/kaggle/input/**/book_recsys/data/__init__.py', recursive=True)
    root = (os.path.dirname(os.path.dirname(os.path.dirname(hits[0]))) if hits
            else '/kaggle/working/book_recsys_src')
    if not hits and not os.path.exists(root):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/MayaDeneva/book_recsys', root], check=True)
    sys.path.insert(0, root)
    for m in [k for k in list(sys.modules) if k == 'book_recsys' or k.startswith('book_recsys.')]:
        del sys.modules[m]
    import book_recsys.data  # noqa: F401
import book_recsys
print('book_recsys at', book_recsys.__file__)

In [ ]:
import glob, os, statistics
import numpy as np
import pandas as pd
import joblib
import torch

from book_recsys.data.schema import BOOK, TS, USER
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.negatives import build_cdf, sample_negatives
from book_recsys.eval.harness import build_relevance
from book_recsys.eval.metrics import mrr, ndcg_at_k, recall_at_k
from book_recsys.llm.fusion import weighted_reciprocal_rank_fusion
from book_recsys.models.content.maxsim import MaxSimRecommender
from book_recsys.models.sequential.recommender import SasRecRecommender
from book_recsys.models.autoencoder.recommender import MultVaeRecommender
from book_recsys.models.autoencoder.train import load_checkpoint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# How many users to score. Lower these if you hit time/memory limits.
EVAL_ACC = 3000   # accuracy (cheap: sampled-neg)
EVAL_POP = 1500   # anti-popularity (heavier: full-catalog top-10)
N_NEG, K = 100, 10

def find_data(name):
    for p in [f'artifacts/{name}', *glob.glob(f'/kaggle/input/**/{name}', recursive=True)]:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name} not in artifacts/ or /kaggle/input/** — attach the dataset')

In [ ]:
# --- same subsample the checkpoint trained on; ORDERED by time for the sequential model ---
sample = pd.read_parquet(find_data('sample.parquet'))
keep = sample[USER].drop_duplicates().sample(30000, random_state=42)
sample = (sample[sample[USER].isin(keep)].sort_values([USER, TS])
          .groupby(USER, sort=False).tail(100).reset_index(drop=True))
sample[BOOK] = sample[BOOK].astype(str)
train, holdout = leave_last_n_out(sample, n=1)
train = train.sort_values([USER, TS])
relevance = build_relevance(holdout)
histories = train.groupby(USER, sort=False)[BOOK].apply(list).to_dict()  # time-ordered
print(f'subsample: {sample[USER].nunique()} users, {len(sample):,} interactions')

In [ ]:
# --- recommenders (all in-process, no RecBole) ---
sas = SasRecRecommender.from_checkpoint(find_data('SASRec_state.pt'),
                                        find_data('SASRec_item_map.json'), device=DEVICE)
catalog = pd.read_parquet(find_data('catalog.parquet'))
catalog[BOOK] = catalog[BOOK].astype(str)
emb = np.load(find_data('embeddings.npy'))
book_ids = catalog[BOOK].tolist()
maxsim = MaxSimRecommender(book_ids, emb)
svd = joblib.load(find_data('models.joblib'))['svd']

vae_model, vae_ckpt = load_checkpoint(find_data('multvae_last.pt'), device=DEVICE)
vae_ids = [str(b) for b in vae_ckpt['ids']]
vae_counts = sample[BOOK].value_counts().reindex(vae_ids).fillna(1.0).to_numpy(dtype=float)
vae_pos = {b: j for j, b in enumerate(vae_ids)}
vae0 = MultVaeRecommender(device=DEVICE, pop_discount=0.0).attach(vae_model, vae_ids, vae_pos, vae_counts)
vae1 = MultVaeRecommender(device=DEVICE, pop_discount=1.0).attach(vae_model, vae_ids, vae_pos, vae_counts)
print(f'SASRec vocab {len(sas._tokens):,} | Mult-VAE vocab {len(vae_ids):,}')

## (A) Accuracy — popularity-matched sampled negatives

Each user's held-out last book is ranked against 100 popularity-matched negatives. This strips the
popularity crutch, so it rewards models that capture *what comes next* over models that just
re-surface bestsellers. (`score_items(history, candidates)` is the same call `FeedService` uses.)

In [ ]:
counts = train[BOOK].value_counts()
pool, cdf = counts.index.to_numpy(), build_cdf(counts.to_numpy())
rng = np.random.default_rng(0)

acc_models = {'SASRec': sas, 'Mult-VAE (a=0)': vae0, 'maxsim': maxsim, 'svd': svd}
ens = [('SASRec+maxsim (RRF)', 'SASRec', 'maxsim'),
       ('Mult-VAE+maxsim (RRF)', 'Mult-VAE (a=0)', 'maxsim')]
agg = {m: {'recall@10': [], 'ndcg@10': [], 'mrr': []}
       for m in list(acc_models) + [n for n, _, _ in ens]}

users = [u for u in relevance if u in histories][:EVAL_ACC]
for n, u in enumerate(users):
    positive = next(iter(relevance[u]))
    hist = histories.get(u, [])
    seen = set(hist) | {positive}
    cands = [positive] + sample_negatives(pool, set(seen), N_NEG, rng, cdf)
    orders = {}
    for name, mdl in acc_models.items():
        sc = mdl.score_items(hist, cands)
        orders[name] = [cands[j] for j in np.argsort(sc)[::-1]]
        agg[name]['recall@10'].append(recall_at_k(orders[name], {positive}, K))
        agg[name]['ndcg@10'].append(ndcg_at_k(orders[name], {positive}, K))
        agg[name]['mrr'].append(mrr(orders[name], {positive}))
    for ens_name, a, b in ens:
        fused = weighted_reciprocal_rank_fusion([(orders[a], 1.0), (orders[b], 1.0)])
        agg[ens_name]['recall@10'].append(recall_at_k(fused, {positive}, K))
        agg[ens_name]['ndcg@10'].append(ndcg_at_k(fused, {positive}, K))
        agg[ens_name]['mrr'].append(mrr(fused, {positive}))
    if (n + 1) % 1000 == 0:
        print(f'  acc {n + 1}/{len(users)}', flush=True)

acc = (pd.DataFrame({m: {k: statistics.fmean(v) for k, v in d.items()} for m, d in agg.items()})
       .T.sort_values('ndcg@10', ascending=False).round(4))
acc.to_csv('study_sasrec_vs_vae_accuracy.csv')
print(f'popularity-matched sampled-neg, {len(users)} users:')
acc

## (B) Anti-popularity diagnostic — full-catalog top-10

Now each model ranks the **whole catalog** and returns its top-10 (excluding the user's history).
We measure how popularity-biased those recommendations are:

- **mean_pop_pct** — average popularity percentile of recommended books (1.0 = always the single most
  popular book; lower = more long-tail).
- **catalog_coverage** — share of the model's own vocabulary it ever recommends across users.
- **gini_freq** — 0 = even spread across the catalog, 1 = a handful of books dominate every feed.

Mult-VAE is shown at α=0 (accuracy-tuned) and α=1 (popularity-discounted / the served config) so you
can see the serendipity knob next to SASRec's *un-tuned* popularity behaviour.

In [ ]:
pct = counts.rank(pct=True)
pop_pct = {b: float(pct[b]) for b in counts.index}

def gini(freqs):
    x = np.sort(np.asarray(freqs, dtype=float)); nn = len(x)
    return float((2 * np.arange(1, nn + 1) - nn - 1).dot(x) / (nn * x.sum())) if x.sum() else 0.0

pop_models = {'SASRec': (sas, len(sas._tokens)), 'Mult-VAE (a=0)': (vae0, len(vae_ids)),
              'Mult-VAE (a=1)': (vae1, len(vae_ids)), 'maxsim': (maxsim, len(book_ids))}
pop_users = users[:EVAL_POP]
prows = {}
for name, (mdl, vocab) in pop_models.items():
    rec_pcts, freq = [], {}
    for u in pop_users:
        for b in mdl.recommend(histories.get(u, []), K):
            rec_pcts.append(pop_pct.get(str(b), 0.0))  # cold/unseen -> least popular
            freq[b] = freq.get(b, 0) + 1
    prows[name] = {'mean_pop_pct': round(statistics.fmean(rec_pcts), 4),
                   'median_pop_pct': round(statistics.median(rec_pcts), 4),
                   'catalog_coverage': round(len(freq) / vocab, 4),
                   'gini_freq': round(gini(list(freq.values())), 4)}
    print(f'  done: {name}', flush=True)

pop = pd.DataFrame(prows).T
pop.to_csv('study_sasrec_vs_vae_antipop.csv')
print(f'full-catalog top-10, {len(pop_users)} users:')
pop

### Reading the results

- **(A)** SASRec is expected to top the popularity-matched table (its sequential bias is the home-turf
  signal); the Mult-VAE (α=0) row is the first real head-to-head against the autoencoder on identical
  users/negatives. The RRF rows show whether the autoencoder is a *complementary* partner to SASRec
  (orthogonal signal → ensemble lifts) or a redundant one.
- **(B)** A high `mean_pop_pct` + high `gini` + low `coverage` means a model leans on the popularity
  head; the opposite means it surfaces the long tail. SASRec has **no popularity discount**, so this
  tells you whether its sequential ranking is intrinsically popularity-biased — compare it to the
  Mult-VAE α=0 → α=1 shift, which is the autoencoder's explicit anti-popularity knob.

CSVs written to the working dir: `study_sasrec_vs_vae_accuracy.csv`, `study_sasrec_vs_vae_antipop.csv`.